# Basis Optimization Demo (TS1)

Load the TS1 mesh and MCF skeleton, run `BasisOptimizer`, and visualize
before/after.

**Requirements:** `mascaf`

In [1]:
import logging
from pathlib import Path

from mascaf import (
    BasisOptimizer,
    BasisOptimizerOptions,
    MeshManager,
    MorphologyGraph,
    SkeletonGraph,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## Paths

In [2]:
ROOT = Path.cwd().parent
mesh_path = ROOT / "data" / "mesh" / "processed" / "TS1.obj"
skeleton_path = ROOT / "data" / "mcf_skeletons" / "TS1_qst0.5_mcst5.polylines.txt"

print(f"Mesh:     {mesh_path}")
print(f"Skeleton: {skeleton_path}")

Mesh:     c:\Users\MainUser\Documents\Repos\mascaf\data\mesh\processed\TS1.obj
Skeleton: c:\Users\MainUser\Documents\Repos\mascaf\data\mcf_skeletons\TS1_qst0.5_mcst5.polylines.txt


## Load mesh and skeleton

In [3]:
mm = MeshManager(mesh_path=str(mesh_path))
skeleton = SkeletonGraph.from_txt(str(skeleton_path))

diagonal = mm.bounding_box_diagonal()
print(f"Bounding box diagonal: {diagonal:.4f}")
print(f"Skeleton: {skeleton.number_of_nodes()} nodes")

INFO: Loaded mesh: 2378 vertices, 4788 faces


Bounding box diagonal: 2710.4504
Skeleton: 376 nodes


## Build morphology basis

Resample the skeleton to a morphology basis for optimization
(`max_edge_length` = 2 % of the bounding-box diagonal).

In [4]:
max_edge_length = 0.05 * diagonal
basis = MorphologyGraph.from_skeleton_graph_resample(skeleton, max_edge_length)
print(
    f"Basis: {basis.number_of_nodes()} nodes, "
    f"{basis.number_of_edges()} edges "
    f"(max_edge_length={max_edge_length:.4f})"
)

# check for outside nodes
outside_ids = basis.get_outside_nodes(mm)
print(f"Found {len(outside_ids)} vertices outside!")

Basis: 88 nodes, 94 edges (max_edge_length=135.5225)
Found 5 vertices outside!


## Visualize mesh with initial basis

In [ ]:
fig = mm.visualize_mesh_3d(
    skel=basis, 
    show_axes=False, 
    title="Initial basis",
    skel_marker_size=2.0,
    skel_line_width=2.0)
fig.show()

## Run BasisOptimizer

In [6]:
basis_options = BasisOptimizerOptions(
    do_pruning=False,
    do_snapping=True,
    do_forcing=True,
    max_iterations=100,
    step_scale=0.5,
    lambda_smooth=0.2,
    lambda_vertex=0.3,
    preserve_terminal_nodes=True,
    step_cap_factor=0.5,
)

optimizer = BasisOptimizer(basis, mm.mesh, basis_options)
optimized = optimizer.optimize()
stats = optimizer.get_optimization_stats()

outside_ids = optimized.get_outside_nodes(mm.mesh)
print(f"Found {len(outside_ids)} vertices outside!")
if len(outside_ids)>0:
    for id in outside_ids:
        print(id, optimized.get_node_position(id))

print("Basis optimization statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

    

INFO: Starting basis optimization...
INFO:   Nodes: 88
INFO: Phase 1 - Snapping: 5 nodes outside mesh
INFO: Phase 2 - Forcing: max 100 iterations
INFO:   Forcing lambdas: smooth=0.2000 vertex=0.3000 repulsion_radius=132.257948
INFO:   Iteration 0: avg movement = 0.221776
INFO:   Iteration 1: avg movement = 0.217412
INFO:   Iteration 2: avg movement = 0.215691
INFO:   Iteration 3: avg movement = 0.212882
INFO:   Iteration 4: avg movement = 0.209690
INFO:   Iteration 5: avg movement = 0.206574
INFO:   Iteration 6: avg movement = 0.203317
INFO:   Iteration 7: avg movement = 0.200598
INFO:   Iteration 8: avg movement = 0.197089
INFO:   Iteration 9: avg movement = 0.194649
INFO:   Iteration 10: avg movement = 0.191321
INFO:   Iteration 11: avg movement = 0.189033
INFO:   Iteration 12: avg movement = 0.187359
INFO:   Iteration 13: avg movement = 0.184512
INFO:   Iteration 14: avg movement = 0.181289
INFO:   Iteration 15: avg movement = 0.178563
INFO:   Iteration 16: avg movement = 0.176098
I

Found 0 vertices outside!
Basis optimization statistics:
  num_nodes: 88
  num_edges: 94
  num_terminal_nodes: 17
  num_branch_nodes: 23
  total_length: 8991.187225219828
  nodes_outside_mesh: 0


## Visualize mesh with optimized basis

In [13]:
fig = mm.visualize_mesh_3d(
    skel=[basis, optimized], 
    show_axes=False, 
    title="Optimized basis",
    skel_marker_size=2.0,
    skel_line_width=2.0,
    skel_color=['red', 'blue']
)
fig.show()